# Decision Tree Training Notebook

This notebook trains a Decision Tree model on the preprocessed CICEVSE2024 dataset for Multiclass classification. It includes data loading, comprehensive data visualisation, hyperparameter tuning, evaluation, and feature importance analysis.

**Dataset**: CICEVSE2024 Network Traffic (14 attack classes + benign)  
**Algorithm**: DecisionTreeClassifier (scikit-learn)  
**Task**: Multiclass classification of EV charging network intrusions

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Plotting config
%matplotlib inline
plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid', palette='deep')

## 1. Load Data
Load the preprocessed train, validation, and test datasets.

In [ ]:
# Adjust path
if os.path.exists('../../../data/processed'):
    DATA_DIR = '../../../data/processed'
elif os.path.exists('data/processed'):
    DATA_DIR = 'data/processed'
else:
    DATA_DIR = '../data/processed'

print(f"Using data directory: {DATA_DIR}")

X_train = pd.read_csv(os.path.join(DATA_DIR, "X_train.csv"))
y_train = pd.read_csv(os.path.join(DATA_DIR, "y_train.csv"))
X_val = pd.read_csv(os.path.join(DATA_DIR, "X_val.csv"))
y_val = pd.read_csv(os.path.join(DATA_DIR, "y_val.csv"))
X_test = pd.read_csv(os.path.join(DATA_DIR, "X_test.csv"))
y_test = pd.read_csv(os.path.join(DATA_DIR, "y_test.csv"))

y_train_multi = y_train["Label_Multiclass"].values.ravel()
y_val_multi = y_val["Label_Multiclass"].values.ravel()
y_test_multi = y_test["Label_Multiclass"].values.ravel()

print("Data loaded successfully!")
print(f"X_train shape: {X_train.shape}")

## 1.1 Dataset Overview

In [ ]:
print(f"X_train shape : {X_train.shape}")
print(f"X_val shape   : {X_val.shape}")
print(f"X_test shape  : {X_test.shape}")
print(f"\nNumber of features: {X_train.shape[1]}")
print(f"\nData types:\n{X_train.dtypes.value_counts()}")
print(f"\nMissing values per column (if any):")
missing = X_train.isnull().sum()
missing_cols = missing[missing > 0]
if len(missing_cols) == 0:
    print("  None — all features are clean.")
else:
    print(missing_cols)

print("\nSummary Statistics (first 10 features):")
X_train.iloc[:, :10].describe().round(3)

## 1.2 Train / Validation / Test Split Sizes

In [ ]:
split_sizes = pd.DataFrame({
    'Split': ['Train', 'Validation', 'Test'],
    'Samples': [len(X_train), len(X_val), len(X_test)]
})
split_sizes['Percentage'] = (split_sizes['Samples'] / split_sizes['Samples'].sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(8, 3))
bars = ax.barh(split_sizes['Split'], split_sizes['Samples'], color=['#2196F3', '#FF9800', '#4CAF50'])
for bar, pct in zip(bars, split_sizes['Percentage']):
    ax.text(bar.get_width() + 5000, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():,.0f} ({pct}%)', va='center', fontsize=11)
ax.set_xlabel('Number of Samples')
ax.set_title('Train / Validation / Test Split Sizes')
plt.tight_layout()
plt.show()

## 1.3 Visualize Class Distributions

In [ ]:
multi_counts = y_train['Label_Multiclass'].value_counts()
colors = sns.color_palette('viridis', len(multi_counts))

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(multi_counts.index, multi_counts.values, color=colors)
ax.set_title('Multiclass Distribution (Train)', fontsize=14)
ax.set_xlabel('Count')
ax.set_ylabel('Attack Type')
for i, val in enumerate(multi_counts.values):
    ax.text(val + 1000, i, f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\nMulticlass label counts:")
print(multi_counts.to_string())

## 1.4 Feature Correlation Heatmap

In [ ]:
top_features = X_train.var().nlargest(30).index.tolist()
corr_matrix = X_train[top_features].corr()

plt.figure(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, fmt='.1f',
            cbar_kws={'shrink': 0.8, 'label': 'Pearson Correlation'})
plt.title('Feature Correlation Heatmap (Top 30 by Variance)', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

## 1.5 Feature Distribution Box Plots

In [ ]:
top10 = X_train.var().nlargest(10).index.tolist()

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, col in enumerate(top10):
    sample = X_train[col].sample(n=min(10000, len(X_train)), random_state=42)
    axes[i].boxplot(sample.values, vert=True, patch_artist=True,
                    boxprops=dict(facecolor='#FF7043', alpha=0.7))
    axes[i].set_title(col, fontsize=9, fontweight='bold')
    axes[i].tick_params(axis='x', labelbottom=False)

plt.suptitle('Top 10 Features by Variance — Box Plots (Scaled Data)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 2. Hyperparameter Tuning
We tune `max_depth` and `min_samples_split` using cross-validation with `GridSearchCV`.

In [ ]:
print("Starting Hyperparameter Tuning for Decision Tree...")
param_grid = {
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10]
}

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    param_grid,
    cv=3,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=2
)

dt_grid.fit(X_train, y_train_multi)

print(f"\nBest Parameters: {dt_grid.best_params_}")
print(f"Best CV F1-Score (Macro): {dt_grid.best_score_:.4f}")

best_model = dt_grid.best_estimator_
print(f"Tree depth: {best_model.get_depth()}, Leaves: {best_model.get_n_leaves()}")

## 3. Evaluation

In [ ]:
def evaluate_model(model, X, y, title_prefix=""):
    preds = model.predict(X)
    
    acc = accuracy_score(y, preds)
    f1_macro = f1_score(y, preds, average='macro', zero_division=0)
    f1_weighted = f1_score(y, preds, average='weighted', zero_division=0)
    
    print(f"--- {title_prefix} ---")
    print(f"Accuracy       : {acc:.4f}")
    print(f"Macro F1-Score : {f1_macro:.4f}")
    print(f"Weighted F1    : {f1_weighted:.4f}")
    print(f"\nClassification Report:")
    print(classification_report(y, preds, zero_division=0))
    
    cm = confusion_matrix(y, preds)
    fig_size = max(8, len(np.unique(y)) * 0.8)
    plt.figure(figsize=(fig_size + 2, fig_size))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges")
    plt.title(f"{title_prefix} Confusion Matrix")
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.show()
    return preds

In [ ]:
_ = evaluate_model(best_model, X_val, y_val_multi, title_prefix="Decision Tree — Validation Set")

In [ ]:
y_pred_test = evaluate_model(best_model, X_test, y_test_multi, title_prefix="Decision Tree — Test Set")

## 4. Feature Importance Analysis
Decision Trees provide Gini-based feature importances. Let's visualise the top 20 most important features.

In [ ]:
importances = best_model.feature_importances_
feature_names = X_train.columns

feat_imp_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

top20 = feat_imp_df.head(20)

fig, ax = plt.subplots(figsize=(10, 8))
colors = sns.color_palette('YlOrRd_r', len(top20))
ax.barh(top20['Feature'][::-1], top20['Importance'][::-1], color=colors[::-1])
ax.set_xlabel('Feature Importance (Gini)', fontsize=12)
ax.set_title('Top 20 Most Important Features — Decision Tree', fontsize=14)
for i, (val, name) in enumerate(zip(top20['Importance'][::-1], top20['Feature'][::-1])):
    ax.text(val + 0.001, i, f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print("\nTop 20 Feature Importances:")
print(top20.to_string(index=False))

## 5. Save Model and Predictions

In [ ]:
if os.path.exists('../../../saved_models'):
    SAVE_DIR = '../../../saved_models'
    PREDS_DIR = '../../../predictions'
elif os.path.exists('saved_models'):
    SAVE_DIR = 'saved_models'
    PREDS_DIR = 'predictions'
else:
    SAVE_DIR = '../saved_models'
    PREDS_DIR = '../predictions'

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(PREDS_DIR, exist_ok=True)

# Save model
model_path = os.path.join(SAVE_DIR, "dt_model_multiclass.pkl")
joblib.dump(best_model, model_path)
print(f"Model saved to {model_path}")

# Save predictions
preds_df = pd.DataFrame({
    'Prediction_Multiclass': y_pred_test,
    'y_true': y_test_multi,
    'y_pred': y_pred_test
})
preds_path = os.path.join(PREDS_DIR, "dt_preds_multiclass.csv")
preds_df.to_csv(preds_path, index=False)
print(f"Predictions saved to {preds_path}")

print("\nDecision Tree pipeline complete!")